In [1]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

I0000 00:00:1773751415.043496  102347 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773751415.079260  102347 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773751416.743328  102347 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [8]:
## load all models 

model = load_model('model.h5')

## load the encoders and scaler

with open('one_hot_encoder.pkl','rb') as file :
    label_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl','rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler = pickle.load(file)

In [9]:
input_data = {

    "CreditScore" : 600,
    "Geography" : 'France',
    "Gender" : 'Female',
    "Age":45,
    "Tenure":12,
    "Balance":70000,
    "NumOfProducts":2,
    "HasCrCard" : 2,
    "IsActiveMember" :1,
    "EstimatedSalary":120000
}

In [14]:
geo_encoded = label_encoder_geo.transform([[input_data['Geography']]]).toarray()

geo_encoded_df = pd.DataFrame(geo_encoded, columns=label_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df

/home/oasys/annclassification/enb/lib/python3.11/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [18]:
input_data_df = pd.DataFrame([input_data])

In [19]:
#encode categorical 
input_data_df['Gender'] = label_encoder_gender.transform(input_data_df['Gender'])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,0,45,12,70000,2,2,1,120000


In [20]:
# conacatenation with onehot encoding data

input_data_df = pd.concat([input_data_df.drop('Geography', axis=1),geo_encoded_df],axis=1)

input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,0,45,12,70000,2,2,1,120000,1.0,0.0,0.0


In [21]:

#scaling the input data

input_scalled = scaler.transform(input_data_df)

In [22]:
input_scalled

array([[-0.53598516, -1.09499335,  0.58015577,  2.42782596, -0.09770129,
         0.80843615,  2.83875637,  0.97481699,  0.34023471,  1.00150113,
        -0.57946723, -0.57638802]])

In [23]:
prediction = model.predict(input_scalled)

prediction

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 60ms/step


array([[0.13622689]], dtype=float32)

In [24]:
prediction_prob = prediction[0][0]

prediction_prob

np.float32(0.13622689)

In [25]:
if prediction_prob > 0.5:
    print("The customer is likely to churn")
else:
    print("the customer is exited")

the customer is exited
